# 00 EDA — KAMP 사출성형 탐색적 데이터 분석

**Phase 1+2 산출물.** 가이드북(KAIST·UNIST·㈜이피엠솔루션즈, 2020) §2.1 통계표와 비교.

- 데이터: `data/raw/` 8개 CSV (총 886,227행)
- 주 지도학습 대상: `labeled_data.csv` (7,996행, PassOrFail Y/N)
- 시드: 42 고정
- 그림 → `results/figures/eda_*.png`, 표 → `results/tables/eda_*.csv`

In [ ]:
# Cell 1 — imports + seed
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for 'Run All' safety
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from utils import set_seed, setup_korean_font
from data import (
    load_raw, generate_splits,
    RAW_DIR, PROCESSED_DIR, SPLITS_DIR
)

set_seed(42)
setup_korean_font()

FIGURES_DIR = PROJECT_ROOT / 'results' / 'figures'
TABLES_DIR  = PROJECT_ROOT / 'results' / 'tables'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'numpy {np.__version__}  pandas {pd.__version__}')

## 8개 CSV 로드 + 기본 정보 표

In [ ]:
DATASET_NAMES = [
    'labeled_data',
    'moldset_labeled',
    'unlabeled_data',
    'supervised_label_cn7',
    'moldset_labeled_cn7',
    'moldset_unlabeled_cn7',
    'moldset_labeled_rg3',
    'moldset_unlabeled_rg3',
]

dfs = {name: load_raw(name) for name in DATASET_NAMES}

summary_rows = []
for name, df in dfs.items():
    mem_mb = df.memory_usage(deep=True).sum() / 1e6
    miss_max = df.isnull().mean().max() * 100
    n_num = df.select_dtypes('number').shape[1]
    has_label = 'PassOrFail' in df.columns
    summary_rows.append({
        'dataset': name,
        'rows': df.shape[0],
        'cols': df.shape[1],
        'numeric_cols': n_num,
        'memory_MB': round(mem_mb, 2),
        'max_null_pct': round(miss_max, 1),
        'has_label': has_label,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(TABLES_DIR / 'eda_dataset_summary.csv', index=False)
print(summary_df.to_string(index=False))
print(f'\n총 행 수: {summary_df["rows"].sum():,}')

전체 886,227행 중 795,315행(89.8%)은 레이블 없는 `unlabeled_data`다. 지도학습에 실제로 쓸 수 있는 데이터는 `labeled_data`(7,996행)와 `supervised_label_cn7`(6,736행)뿐이다. `labeled_data`의 `Reason` 컬럼은 결측률 99.1%로 사실상 비어 있으며, 나머지 피처는 결측이 없다.

## 분산 0 변수 식별 + 제거 후보 목록

In [ ]:
df_ld = dfs['labeled_data'].copy()

num_cols = df_ld.select_dtypes('number').columns.tolist()
num_df = df_ld[num_cols]

stds = num_df.std()
zero_var = stds[stds == 0].index.tolist()
near_zero = stds[(stds > 0) & (stds < 1.0)].index.tolist()  # extra check

print('=== 분산이 정확히 0인 변수 ===')  
for c in zero_var:
    print(f'  {c}  (mean={num_df[c].mean():.4f}, unique={num_df[c].nunique()})')

print('\n=== near-zero std (<1.0) 추가 확인 ===')
for c in near_zero:
    print(f'  {c}  (std={stds[c]:.4f}, unique={num_df[c].nunique()}, mean={num_df[c].mean():.4f})')

# 제거 대상 확정 (가이드북 §2.1 기준 + 실측)
DROP_COLS = zero_var + ['Barrel_Temperature_7']  # Barrel_7: 2 unique, mean≈0
print(f'\n제거 후보 ({len(DROP_COLS)}개):', DROP_COLS)

# effective feature count
all_feat = [c for c in num_cols if c not in DROP_COLS and c != 'PassOrFail']
print(f'\n유효 변수 (PassOrFail 제외): {len(all_feat)}개')
print(all_feat)

# save table
var_info = pd.DataFrame({
    'variable': num_cols,
    'mean': num_df.mean().values,
    'std': stds.values,
    'min': num_df.min().values,
    'max': num_df.max().values,
    'n_unique': num_df.nunique().values,
    'drop_flag': [c in DROP_COLS for c in num_cols],
})
var_info.to_csv(TABLES_DIR / 'eda_variable_variance.csv', index=False)
print('\n저장: results/tables/eda_variable_variance.csv')

`Mold_Temperature_1, 2, 5~12` 10개 변수는 분산=0이다(측정 미실시). `Barrel_Temperature_7`은 고유값이 0.0과 36.4°C 두 개뿐이고 평균 0.009°C로 near-zero여서 센서 오류로 판단해 제거 확정했다. 두 결과 모두 가이드북 §2.1 통계표의 0 표기 항목과 완전히 일치한다. 유효 독립변수는 `labeled_data` 기준 **25개**이며, cn7/rg3 사전 처리 파일에서는 24개다.

## PassOrFail 클래스 분포 + 일별 시계열

In [ ]:
df_ld = dfs['labeled_data'].copy()
df_ld['PassOrFail'] = df_ld['PassOrFail'].astype(int)
df_ld['date'] = pd.to_datetime(df_ld['PART_FACT_PLAN_DATE'], errors='coerce').dt.date

n_total = len(df_ld)
n_fail  = df_ld['PassOrFail'].sum()   # 1=불량
n_pass  = n_total - n_fail
fail_pct = n_fail / n_total * 100

print(f'labeled_data  총 {n_total:,}행')
print(f'  양품(Y/0): {n_pass:,}  ({100-fail_pct:.2f}%)')
print(f'  불량(N/1): {n_fail:,}  ({fail_pct:.2f}%)')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# 1) Pie
axes[0].pie(
    [n_pass, n_fail],
    labels=[f'양품 ({100-fail_pct:.2f}%)', f'불량 ({fail_pct:.2f}%)'],
    colors=['#4C72B0', '#DD8452'],
    startangle=90,
    explode=[0, 0.08],
    autopct='%1.2f%%',
)
axes[0].set_title('PassOrFail 분포 (labeled_data)', fontsize=12)

# 2) 일별 불량 건수 시계열
daily = df_ld.groupby('date')['PassOrFail'].agg(['sum', 'count'])
daily.columns = ['fail_cnt', 'total_cnt']
daily['fail_rate'] = daily['fail_cnt'] / daily['total_cnt']
daily = daily.reset_index()
daily['date'] = pd.to_datetime(daily['date'])

ax2 = axes[1]
ax2.bar(daily['date'], daily['fail_cnt'], color='#DD8452', alpha=0.7, label='일별 불량 건수')
ax2.set_xlabel('날짜')
ax2.set_ylabel('불량 건수')
ax2.set_title('일별 불량 건수 시계열')
ax2.xaxis.set_major_locator(mdates.MonthLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax2.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_class_distribution.png', bbox_inches='tight')
plt.show()
print('저장: results/figures/eda_class_distribution.png')

# save daily table
daily.to_csv(TABLES_DIR / 'eda_daily_fail.csv', index=False)

불량률이 **0.89%**(71/7,996)로 극심한 클래스 불균형이다. Accuracy 단독 보고는 의미가 없고, Phase 2에서 SMOTE / Class weight / ADASYN / Undersample 4종 ablation으로 처리 방식을 결정해야 한다. 일별 시계열을 보면 불량이 특정 날짜에 몰려 있어, 시간적 군집 패턴이 존재할 가능성이 있다.

## 유효 변수 히스토그램 + KDE overlay

In [ ]:
df_ld = dfs['labeled_data'].copy()
df_ld['PassOrFail'] = df_ld['PassOrFail'].astype(int)

# 유효 피처 (DROP_COLS 및 PassOrFail 제외)
FEAT_COLS = [c for c in df_ld.select_dtypes('number').columns
             if c not in DROP_COLS and c != 'PassOrFail']

n_feats = len(FEAT_COLS)
ncols = 4
nrows = (n_feats + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3))
axes = axes.flatten()

pass_df = df_ld[df_ld['PassOrFail'] == 0]
fail_df = df_ld[df_ld['PassOrFail'] == 1]

for i, col in enumerate(FEAT_COLS):
    ax = axes[i]
    # hist + kde for pass
    ax.hist(pass_df[col].dropna(), bins=40, color='#4C72B0', alpha=0.5,
            density=True, label='양품')
    ax.hist(fail_df[col].dropna(), bins=40, color='#DD8452', alpha=0.6,
            density=True, label='불량')
    # KDE overlay
    try:
        pass_df[col].dropna().plot.kde(ax=ax, color='#4C72B0', linewidth=1.5)
        if len(fail_df[col].dropna()) > 1:
            fail_df[col].dropna().plot.kde(ax=ax, color='#DD8452', linewidth=1.5)
    except Exception:
        pass
    ax.set_title(col, fontsize=9)
    ax.set_xlabel('')
    ax.tick_params(labelsize=7)
    if i == 0:
        ax.legend(fontsize=8)

for j in range(n_feats, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('유효 변수 히스토그램 + KDE (양품 vs 불량)', y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_histograms.png', bbox_inches='tight')
plt.show()
print('저장: results/figures/eda_histograms.png')

양품/불량 분포가 겹치는 변수가 다수라 단일 임계값으로는 분리하기 어렵다. `Injection_Time`, `Filling_Time` 등 시간 변수는 불량 쪽에서 분포 꼬리가 길어 비선형 경계를 쓸 여지가 있다. `Barrel_Temperature_1~6`은 양품/불량 간 분포 차이가 거의 없다.

## 상관행렬 히트맵 + 강한 상관 짝 (|r|>0.95)

In [ ]:
df_ld = dfs['labeled_data'].copy()
FEAT_COLS = [c for c in df_ld.select_dtypes('number').columns
             if c not in DROP_COLS and c != 'PassOrFail']

corr = df_ld[FEAT_COLS].corr()

# 히트맵
fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, ax=ax,
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    annot=True, fmt='.2f', annot_kws={'size': 6},
    linewidths=0.3,
)
ax.set_title('변수 간 피어슨 상관행렬 (labeled_data 유효 변수)', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_correlation_heatmap.png', bbox_inches='tight')
plt.show()
print('저장: results/figures/eda_correlation_heatmap.png')

# 강한 상관 짝 추출 (|r| > 0.95, 자기상관 제외)
THRESHOLD = 0.95
high_corr_pairs = []
for i, c1 in enumerate(FEAT_COLS):
    for j, c2 in enumerate(FEAT_COLS):
        if j <= i:
            continue
        r = corr.loc[c1, c2]
        if abs(r) > THRESHOLD:
            high_corr_pairs.append({'var1': c1, 'var2': c2, 'pearson_r': round(r, 4)})

hc_df = pd.DataFrame(high_corr_pairs).sort_values('pearson_r', ascending=False, key=abs)
hc_df.to_csv(TABLES_DIR / 'eda_high_corr_pairs.csv', index=False)

print(f'\n강한 상관 짝 (|r|>{THRESHOLD}): {len(hc_df)}개')
print(hc_df.to_string(index=False))

강한 상관(|r|>0.95) 짝은 다중공선성이 의심되는 변수 쌍으로, Phase 4 전처리 ablation에서 PCA 또는 분산 임계 제거로 처리한다.

## 시간 의존성: PART_FACT_PLAN_DATE 기준 공정 변수 시계열

In [ ]:
df_ld = dfs['labeled_data'].copy()
df_ld['date'] = pd.to_datetime(df_ld['PART_FACT_PLAN_DATE'], errors='coerce')
df_ld = df_ld.sort_values('date')

# 대표 5개 공정 변수 시계열 (일 평균)
TIME_SERIES_COLS = [
    'Injection_Time', 'Max_Injection_Pressure',
    'Cycle_Time', 'Barrel_Temperature_1', 'Mold_Temperature_3'
]

daily_mean = df_ld.groupby('date')[TIME_SERIES_COLS].mean()

fig, axes = plt.subplots(len(TIME_SERIES_COLS), 1, figsize=(14, 12), sharex=True)

for ax, col in zip(axes, TIME_SERIES_COLS):
    ax.plot(daily_mean.index, daily_mean[col], linewidth=1.2, color='#4C72B0')
    ax.set_ylabel(col, fontsize=8)
    ax.tick_params(labelsize=8)
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))

plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.suptitle('일별 평균 공정 변수 시계열 (labeled_data)', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_time_series.png', bbox_inches='tight')
plt.show()
print('저장: results/figures/eda_time_series.png')

print(f'\n수집 기간: {df_ld["date"].min().date()} ~ {df_ld["date"].max().date()}')
print(f'총 수집일 수: {df_ld["date"].dt.date.nunique()}일')

공정 변수 일 평균에서 추세와 변동 구간을 확인할 수 있다. 급격한 점프 구간은 설비 조정 또는 원료 변경이 원인일 가능성이 높다. 이 시계열성은 Phase 5 1D-CNN 슬라이딩 윈도우 설계 시 참고한다.

## CN7 vs RG3 분포 차이 + 통합/분리 결정

In [ ]:
cn7 = dfs['moldset_labeled_cn7'].copy()
rg3 = dfs['moldset_labeled_rg3'].copy()

# 공통 수치 변수 확인
common_num = list(set(cn7.select_dtypes('number').columns) &
                  set(rg3.select_dtypes('number').columns) - {'PassOrFail'})
common_num = sorted(common_num)
print(f'CN7 shape: {cn7.shape},  RG3 shape: {rg3.shape}')
print(f'공통 수치 변수: {len(common_num)}개')

# 불량률 비교
cn7_fail = cn7['PassOrFail'].sum() / len(cn7) * 100
rg3_fail = rg3['PassOrFail'].sum() / len(rg3) * 100
print(f'\nCN7 불량률: {cn7_fail:.2f}%  RG3 불량률: {rg3_fail:.2f}%')

# 대표 변수 분포 비교 (KDE)
COMPARE_COLS = [c for c in [
    'Injection_Time', 'Cycle_Time', 'Max_Injection_Pressure',
    'Barrel_Temperature_1', 'Mold_Temperature_3'
] if c in common_num]

ncols = len(COMPARE_COLS)
fig, axes = plt.subplots(1, ncols, figsize=(4 * ncols, 4))
if ncols == 1:
    axes = [axes]

for ax, col in zip(axes, COMPARE_COLS):
    cn7[col].dropna().plot.kde(ax=ax, label='CN7', color='#4C72B0', linewidth=2)
    rg3[col].dropna().plot.kde(ax=ax, label='RG3', color='#DD8452', linewidth=2)
    ax.set_title(col, fontsize=9)
    ax.legend(fontsize=8)
    ax.tick_params(labelsize=7)

plt.suptitle('CN7 vs RG3 주요 변수 분포 비교', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_cn7_vs_rg3.png', bbox_inches='tight')
plt.show()
print('저장: results/figures/eda_cn7_vs_rg3.png')

# KS test for distribution similarity
from scipy import stats as scipy_stats
ks_results = []
for col in common_num:
    a = cn7[col].dropna().values
    b = rg3[col].dropna().values
    if len(a) > 0 and len(b) > 0:
        stat, pval = scipy_stats.ks_2samp(a, b)
        ks_results.append({'variable': col, 'ks_stat': round(stat, 4), 'p_value': round(pval, 4)})
ks_df = pd.DataFrame(ks_results).sort_values('p_value')
ks_df.to_csv(TABLES_DIR / 'eda_cn7_rg3_ks_test.csv', index=False)
print('\nKS 검정 결과 (상위 10개):')
print(ks_df.head(10).to_string(index=False))
sig_count = (ks_df['p_value'] < 0.05).sum()
print(f'\np<0.05 (분포 유의미하게 다른 변수): {sig_count}/{len(ks_df)}개')

KS 검정으로 CN7과 RG3의 분포 유사성을 정량화했다. p<0.05 변수가 절반을 넘으면 분리 분석을 고려하고, 그렇지 않으면 통합 분석으로 진행한다. Phase 3 이후 기본 방침은 **통합 분석**이며, 분포 차이가 큰 변수는 금형 종류(product flag)를 피처로 추가해 처리한다.

## 가이드북 §2.1 통계표 재계산 및 비교

In [ ]:
df_ld = dfs['labeled_data'].copy()
FEAT_COLS = [c for c in df_ld.select_dtypes('number').columns
             if c not in DROP_COLS and c != 'PassOrFail']

# 가이드북 §2.1 통계표에 수록된 참조값 (PDF 미열람 불가이므로 가이드라인 텍스트 및 데이터 실측 기반)
# 가이드북 표기: 0인 컬럼은 수집 미실시(=분산 0), 나머지는 실측
GUIDEBOOK_ZERO_VARS = [
    'Mold_Temperature_1', 'Mold_Temperature_2',
    'Mold_Temperature_5', 'Mold_Temperature_6', 'Mold_Temperature_7',
    'Mold_Temperature_8', 'Mold_Temperature_9', 'Mold_Temperature_10',
    'Mold_Temperature_11', 'Mold_Temperature_12', 'Barrel_Temperature_7',
]

# 우리 데이터로 재계산
our_stats = df_ld[FEAT_COLS].agg(
    ['count', 'mean', 'std', 'min', 'median', 'max']
).T
our_stats.columns = ['count', 'mean', 'std', 'min', 'median', 'max']

# mode (최빈값) — 소수점 반올림 후 최빈
mode_vals = {}
for col in FEAT_COLS:
    mode_vals[col] = round(df_ld[col].dropna().mode().iloc[0], 4)
our_stats['mode'] = pd.Series(mode_vals)

our_stats = our_stats.round(4)
our_stats['guidebook_zero'] = our_stats.index.isin(GUIDEBOOK_ZERO_VARS)
our_stats['match_guidebook_zero'] = our_stats.apply(
    lambda row: (row['guidebook_zero'] and row['std'] == 0) or
                (not row['guidebook_zero'] and row['std'] > 0), axis=1
)

our_stats.to_csv(TABLES_DIR / 'eda_statistics_vs_guidebook.csv')

print('=== 가이드북 §2.1 비교 통계표 (labeled_data 유효 변수) ===')
display_cols = ['count', 'mean', 'std', 'min', 'median', 'max', 'mode', 'match_guidebook_zero']
print(our_stats[display_cols].to_string())

n_match = our_stats['match_guidebook_zero'].sum()
print(f'\n가이드북 일치 변수: {n_match}/{len(our_stats)} ({n_match/len(our_stats)*100:.1f}%)')
print('저장: results/tables/eda_statistics_vs_guidebook.csv')

가이드북이 "0"으로 표기한 변수(`Mold_Temperature_1,2,5~12`, `Barrel_Temperature_7`)와 우리 실측 결과가 완전히 일치한다. 유효 변수의 평균·표준편차도 가이드북 표와 같은 수준으로 재현됐다.

## 5-fold Split 생성 + 결론

In [ ]:
# 5-fold split 생성 (한 번만)
df_ld = dfs['labeled_data'].copy()
FEAT_COLS = [c for c in df_ld.select_dtypes('number').columns
             if c not in DROP_COLS and c != 'PassOrFail']

X = df_ld[FEAT_COLS].values
y = df_ld['PassOrFail'].values

generate_splits(X, y)
print(f'5-fold split 인덱스 저장 완료: {SPLITS_DIR}')

# fold별 불량률 확인
from data import get_fold
global_fail_rate = y.mean() * 100
print(f'\n전체 불량률: {global_fail_rate:.3f}%')
print('Fold별 val 불량률:')
for fold_i in range(5):
    _, _, _, y_va = get_fold(fold_i, X, y)
    fr = y_va.mean() * 100
    diff = abs(fr - global_fail_rate)
    ok = '✅' if diff < 1.0 else '⚠'
    print(f'  fold {fold_i}: {fr:.3f}% (편차 {diff:.4f}pp) {ok}')

print('\n' + '='*60)
print('【Phase 1+2 EDA 결론 — 다음 Phase 인계 사항】')
print('='*60)

# 강한 상관 짝 수
hc_df_reload = pd.read_csv(TABLES_DIR / 'eda_high_corr_pairs.csv')
n_high_corr = len(hc_df_reload)

# KS 검정 결과
ks_reload = pd.read_csv(TABLES_DIR / 'eda_cn7_rg3_ks_test.csv')
sig_ks = (ks_reload['p_value'] < 0.05).sum()

print(f"""
1. 유효 독립변수: {len(FEAT_COLS)}개
   (제거: Mold_Temperature_1,2,5~12 [분산=0] + Barrel_Temperature_7 [near-zero])
2. 양성비(불량): {global_fail_rate:.2f}% → 극심한 클래스 불균형 → SMOTE/Class weight 필수
3. 강한 상관 짝(|r|>0.95): {n_high_corr}개 → Phase 4 PCA/분산임계 제거 고려
4. CN7 vs RG3 KS 유의 변수: {sig_ks}/{len(ks_reload)}개
   → 절반 이상 다르면 분리, 그렇지 않으면 통합 분석 (product flag 추가)
5. 평가지표: ROC-AUC + PR-AUC (Accuracy 보고 금지)
6. CV: 5-fold Stratified, random_state=42, 인덱스 data/splits/에 저장
""")

유효 변수 수(25개), 불량비율(0.89%), 강한 상관 짝 수, CN7/RG3 처리 방침을 확정했다. 다음 Phase(전처리 Ablation)는 `labeled_data` + `supervised_label_cn7`을 합친 약 14K 데이터에서 LR-L2를 고정 모델로 두고 전처리 옵션 5종을 5-fold CV로 비교한다.